In [17]:
%pip install s3fs python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os
import uuid
import random
from dotenv import load_dotenv

In [19]:
load_dotenv()

True

In [20]:
rideIDs = []
driverIDs = []

In [21]:
#Config S3 bucket for storing data
BUCKET_PATH = os.environ.get('S3_PATH')
print(BUCKET_PATH)

storage_options = {
    "key": os.environ.get('IAM_ACCESS_KEY'),
    "secret": os.environ.get('IAM_SECRET')
}

s3://amzn-s3-prj-bucket


In [22]:
def generate_ride_id_list(size):
    """
    Generate a list of unique ride ids, with the digit being the digit of size  
    """

    
    print("⏳ Đang tạo mảng ID ngẫu nhiên...")
    # np.arange tạo dãy số, np.random.shuffle xáo trộn cực nhanh trên C-level
    all_ids = np.arange(1, size + 1, dtype=np.int64)
    np.random.shuffle(all_ids)
    return all_ids

In [23]:
def get_driver_id(num_records, unique_drivers_ratio=0.01    ):
    num_unique = int(num_records * unique_drivers_ratio)
    
    # 1. Tạo một "bể" ID tài xế duy nhất (dạng số nguyên để tối ưu RAM cho Arch Linux)
    #
    unique_drivers = np.arange(1000, 1000 + num_unique, dtype=np.int64)
    driverIDs.extend(unique_drivers)
    
    # 2. Phân bổ ngẫu nhiên các ID này vào num_records bản ghi
    # Cách này mô phỏng 50/50 hoặc bất kỳ tỉ lệ nào bạn muốn một cách cực nhanh
    driver_column = np.random.choice(unique_drivers, size=num_records)
    return driver_column


In [ ]:
def generate_taxi_data_structured(num_files=12, records_per_file=2000000, start_year=2026, start_month=1):
    print(type(rideIDs))
    rideIDs.extend(generate_ride_id_list(num_files * records_per_file))# so that I can pop data

    current_year = start_year
    current_month = start_month

    for i in range(num_files):
        file_name = f"yellow_tripdata_{current_year}-{current_month:02d}.parquet"
        full_path = f"{BUCKET_PATH}/{file_name}"
        
        print(f"⏳ Đang xử lý {records_per_file} dòng cho {file_name}...")
        # 1. Tạo mảng thời gian ngẫu nhiên (Vectorized)
        start_dt = np.datetime64(f'{current_year}-{current_month:02d}-01')
        # Random số phút trong tháng (xấp xỉ 28 ngày để an toàn cho mọi tháng)
        random_minutes = np.random.randint(0, 28 * 24 * 60, records_per_file)
        pickup_times = start_dt + random_minutes.astype('timedelta64[m]')
        dropoff_times = pickup_times + np.random.randint(5, 60, records_per_file).astype('timedelta64[m]')

        data = {
            "rideID": rideIDs.pop(),
            "VendorID": np.random.choice([1, 2, 6], records_per_file),
            "lpep_pickup_datetime": pickup_times,
            "lpep_dropoff_datetime": dropoff_times,
            "store_and_fwd_flag": np.random.choice(['Y', 'N'], records_per_file),
            "RatecodeID": np.random.choice([1, 2, 3, 4, 5, 6, 99], records_per_file),
            "PULocationID": np.random.randint(1, 263, records_per_file),
            "DOLocationID": np.random.randint(1, 263, records_per_file),
            "passenger_count": np.random.randint(1, 6, records_per_file),
            "trip_distance": np.random.uniform(1.0, 15.0, records_per_file).astype(np.float32),
            "fare_amount": np.random.uniform(5.0, 80.0, records_per_file).astype(np.float32),
            "extra": np.random.choice([0.0, 0.5, 1.0], records_per_file),
            "mta_tax": 0.5,
            "tip_amount": np.random.uniform(0.0, 15.0, records_per_file).astype(np.float32),
            "tolls_amount": np.random.choice([0.0, 6.55], records_per_file),
            "improvement_surcharge": 0.3,
            "congestion_surcharge": 2.75,
            # Phí CBD Relief Zone áp dụng từ 05/01/2025 [cite: 10]
            "cbd_congestion_fee": np.where(pickup_times >= np.datetime64('2025-01-05'), 0.75, 0.0), # 
            "payment_type": np.random.randint(1, 5, records_per_file),
            "trip_type": np.random.choice([1, 2], records_per_file) # [cite: 6]
        }

        data['DriverID'] = get_driver_id(num_records=records_per_file)

        # 3. Chuyển sang DataFrame và tính toán cột tổng
        df = pd.DataFrame(data)
        df["total_amount"] = df[["fare_amount", "extra", "mta_tax", "tip_amount", 
                                 "tolls_amount", "improvement_surcharge", 
                                 "congestion_surcharge", "cbd_congestion_fee"]].sum(axis=1)

        # 4. Upload trực tiếp lên S3
        print(f"📤 Đang upload {file_name}...")
        df.to_csv(full_path, storage_options=storage_options, index=False)
        print(f"✅ Đã upload thành công: {full_path}")

        # Giải phóng bộ nhớ RAM
        del df
        
        # Tăng tháng
        current_month += 1
        if current_month > 12:
            current_month = 1
            current_year += 1


In [25]:
generate_taxi_data_structured(
    num_files=1,         # Tạo 6 file tương ứng với 6 tháng
    records_per_file=67000000, 
    start_year=2022, 
    start_month=1
)

<class 'list'>
⏳ Đang tạo mảng ID ngẫu nhiên...
⏳ Đang xử lý 67000000 dòng cho yellow_tripdata_2022-01.parquet...
📤 Đang upload yellow_tripdata_2022-01.parquet...
✅ Đã upload thành công: s3://amzn-s3-prj-bucket/yellow_tripdata_2022-01.parquet


In [26]:
print(len(rideIDs))
print(len(driverIDs))

66999999
670000


In [27]:
#popular British first name
first_name = ["James", "Oliver", "Benjamin", "Ethan", "Lucas", "Mason", "Logan", "Alexander", "Elijah", "Jacob",
              "Emma", "Olivia", "Ava", "Isabella", "Sophia", "Mia", "Charlotte", "Amelia", "Evelyn", "Abigail", 
              "Emily", "Ella", "Madison", "Scarlett", "Victoria", "Aria", "Grace", "Chloe",]

In [28]:
#popular British last name
last_name = ["Smith", "Johnson", "Williams", "Brown", "Jones", "Garcia", "Miller", "Davis", "Rodriguez", "Martinez", "Hernandez", "Lopez", "Gonzalez", "Wilson", "Anderson", "Thomas", "Taylor", "Moore", "Jackson",
                "White", "Harris", "Martin", "Thompson", "Garcia", "Martinez", "Robinson", "Clark", "Rodriguez", "Lewis", "Lee", "Walker", "Hall", "Allen", "Young", "King", "Wright", "Scott", "Torres", "Nguyen", "Hill", "Flores",
                "Green", "Adams", "Nelson", "Baker", "Hall", "Rivera", "Campbell", "Mitchell", "Carter", "Roberts", "Gomez", "Phillips", "Evans", "Turner", "Diaz", "Parker", "Cruz", "Edwards", "Collins"]

In [29]:
vehicles = ["Ford Escape", "Toyota Camry", "Toyota Sienna", "Honda Accord", "Honda CR-V", "Chevrolet Equinox", "Nissan Rogue", "Hyundai Tucson", "Subaru Outback", "Mazda CX-5",
           "Jeep Grand Cherokee", "Volkswagen Tiguan", "Kia Sorento", "Subaru Forester", "Hyundai Santa Fe", "Mazda CX-9", "Toyota RAV4", "Honda Pilot", "Ford Explorer", "Chevrolet Traverse", "Nissan Murano"]

In [30]:
def generate_drivers():
    """generate drivers based on existing IDs"""
    drivers = []
    unique_driver_ids = list(set(driverIDs))
    for id in unique_driver_ids:
        driver = {
            "DriverID": id,
            "FirstName": random.choice(first_name),
            "LastName": random.choice(last_name),
            "LicenseNumber": str(uuid.uuid4())[:8].upper(),
            "Vehicle": random.choice(vehicles),
            "YearsOfExperience": random.randint(1, 30)
        }
        drivers.append(driver)
    return drivers


In [31]:
%pip install psycopg2-binary sqlalchemy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [33]:
import pandas as pd
from sqlalchemy import create_engine, text
import os
from urllib.parse import quote_plus
import logging
import time

# --- Cấu hình Logging ---
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Giả lập hàm generate_drivers (thay bằng hàm thật của bạn)
drivers_list = generate_drivers()
df_drivers = pd.DataFrame(drivers_list)

# --- Chi tiết kết nối ---
password = os.environ.get('SUPABASE_DB_PASSWORD')
host = "db.qittktngfmjrtgrjdoye.supabase.co"
user = "postgres"
port = "5432"
db_name = "postgres"

if not password:
    logger.error("❌ Biến môi trường SUPABASE_DB_PASSWORD chưa được thiết lập!")
    raise ValueError("❌ SUPABASE_DB_PASSWORD is not set")

# Mã hóa mật khẩu an toàn
encoded_password = quote_plus(password)
db_url = f"postgresql+psycopg2://{user}:{encoded_password}@{host}:{port}/{db_name}"

# Tạo engine với cấu hình tối ưu
engine = create_engine(
    db_url,
    connect_args={"sslmode": "require"},
    pool_pre_ping=True  # Kiểm tra kết nối còn sống trước khi dùng
)

try:
    # 1. Kiểm tra và xác nhận kết nối
    logger.info(f"🔄 Đang kết nối tới Supabase host: {host}...")
    start_time = time.time()
    
    with engine.connect() as conn:
        conn.execute(text("SELECT 1"))
        logger.info("✅ Kết nối thành công! Database đã sẵn sàng.")

    # 2. Bắt đầu quá trình ghi dữ liệu
    with engine.begin() as conn:
        # Xóa dữ liệu cũ
        logger.info("🗑️ Đang thực hiện TRUNCATE table 'driver_test'...")
        conn.execute(text("TRUNCATE TABLE driver_test RESTART IDENTITY;"))
        logger.info("✅ Đã làm trống bảng 'driver_test'.")

        # Ghi dữ liệu mới với Log tiến độ
        total_rows = len(df_drivers)
        chunk_size = 1000
        logger.info(f"🚀 Bắt đầu ghi {total_rows} dòng vào 'driver_test' (chunk_size={chunk_size})...")

        # Hàm callback để log tiến độ cho từng chunk
        def chunk_inserted(pd_table, conn, keys, data_iter):
            data = [dict(zip(keys, row)) for row in data_iter]
            conn.execute(pd_table.table.insert(), data)
            # Log sau mỗi chunk được chèn
            logger.info(f"📑 Đã ghi xong một phân đoạn dữ liệu vào database...")

        df_drivers.to_sql(
            'driver_test',
            conn,
            if_exists='append',
            index=False,
            method='multi', # Hoặc dùng 'chunk_inserted' nếu muốn log chi tiết từng chunk
            chunksize=chunk_size
        )

    end_time = time.time()
    duration = round(end_time - start_time, 2)
    logger.info(f"✨ Hoàn tất! Đã làm mới bảng 'driver' với {total_rows} dòng.")
    logger.info(f"⏱️ Tổng thời gian thực hiện: {duration} giây.")

except Exception as e:
    logger.error("❌ Lỗi xảy ra trong quá trình thao tác với Supabase:")
    logger.error(f"Chi tiết lỗi: {e}")

2026-03-19 21:55:50,441 - INFO - 🔄 Đang kết nối tới Supabase host: db.qittktngfmjrtgrjdoye.supabase.co...
2026-03-19 21:55:53,096 - INFO - ✅ Kết nối thành công! Database đã sẵn sàng.
2026-03-19 21:55:53,384 - INFO - 🗑️ Đang thực hiện TRUNCATE table 'driver_test'...
2026-03-19 21:55:53,679 - INFO - ✅ Đã làm trống bảng 'driver_test'.
2026-03-19 21:55:53,680 - INFO - 🚀 Bắt đầu ghi 670000 dòng vào 'driver_test' (chunk_size=1000)...
2026-03-19 21:58:29,318 - INFO - ✨ Hoàn tất! Đã làm mới bảng 'driver' với 670000 dòng.
2026-03-19 21:58:29,318 - INFO - ⏱️ Tổng thời gian thực hiện: 158.88 giây.
